# Some parameters

In [ ]:
#constant values used all over the code
EPOCHS = 10
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
IMAGE_SIZE_C = (224, 224, 3) #same as image size, but with the channels
LOCAL_DIR = "./ants_bees"

# MODEL 3) External model (Vision Transformer from Hugging Face)

Look here for details:

https://huggingface.co/docs/transformers/model_doc/vit?usage=AutoModel

It's a vision trasformer, so the image is divided in patches, processed in sequence - kinda like tokens for a text transformer.

There is an less straightforward installation here, and if you look into the installation logs you'll see warnings here and there. The example works for the intended purposes, though.

## Installing HF transformers

Here we have a first technical difficulty: Hugging Face, with version 5 and above, moved away from tensorflow background and starte using [JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html) background.

We *could* switch to JAX background in Keras3, but in this notebook for the sake of simplicity we'll try to keep the number of changes to a minimum. So, when installing the transformers library we stick to the last TF compatible version:

In [ ]:
!pip install "transformers<5" tf-keras

We'll also need to adapt our code. The version of Keras that the HF implementation uses is the so called "legacy" Keras. We'll need to set this configuration before any import.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tf_keras as keras
from transformers import TFViTModel

## Loading the HF ViT model

We can finally load the model locally.

In [ ]:
# Load pretrained ViT
base_model = TFViTModel.from_pretrained(
    "google/vit-base-patch16-224",
     use_safetensors=False
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tf_model.h5:   0%|          | 0.00/347M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some layers from the model checkpoint at google/vit-base-patch16-224 were not used when initializing TFViTModel: ['classifier']
- This IS expected if you are initializing TFViTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFViTModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['vit/pooler/dense/kernel:0', 'vit/pooler/dense/bias:0']
You should probably TRAIN this model on a down-stream task to be

## The usual transfer learning wrapper

Keep in mind that (a bit buried) in the ViT documentation we find that the input data should have some preprocessing different from what we are used to:

- images should be normalized in the [-1,1] range
- image dimension should be organized as "channel first", we are "channel last"

In [ ]:

#freeze the pretrained weights
base_model.trainable = False

#the usual Keras sequential model
model = keras.Sequential()

#we setup the data input for the network
model.add(keras.Input(shape=IMAGE_SIZE_C))

#preprocessing A): rescaling from [0,255] to [-1,1]
model.add(keras.layers.Rescaling(
    scale=1./127.5,
    offset=-1
))

#preprocessing B): the expected data order is different
# - Keras standard: (batch, height, width, channel)
# - HF ViT        : (batch, channel, height, width)
model.add(keras.layers.Permute((3, 1, 2)))

#Hugging Face model, invoked via a Lambda layer
model.add(keras.layers.Lambda(
    lambda x: base_model(
        pixel_values=x,
        training=False
    ).last_hidden_state
))

#extract class token
model.add(keras.layers.Lambda(
    lambda x: x[:, 0, :]
))

#minimal classification head
model.add(keras.layers.Dense(1, activation="sigmoid"))

#compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

The following Variables were used a Lambda layer's call (lambda), but
are not present in its tracked objects:
  <tf.Variable 'tf_vi_t_model/vit/embeddings/cls_token:0' shape=(1, 1, 768) dtype=float32>
  <tf.Variable 'tf_vi_t_model/vit/embeddings/position_embeddings:0' shape=(1, 197, 768) dtype=float32>
  <tf.Variable 'tf_vi_t_model/vit/embeddings/patch_embeddings/projection/kernel:0' shape=(16, 16, 3, 768) dtype=float32>
  <tf.Variable 'tf_vi_t_model/vit/embeddings/patch_embeddings/projection/bias:0' shape=(768,) dtype=float32>
  <tf.Variable 'tf_vi_t_model/vit/encoder/layer_._0/attention/attention/query/kernel:0' shape=(768, 768) dtype=float32>
  <tf.Variable 'tf_vi_t_model/vit/encoder/layer_._0/attention/attention/query/bias:0' shape=(768,) dtype=float32>
  <tf.Variable 'tf_vi_t_model/vit/encoder/layer_._0/attention/attention/key/kernel:0' shape=(768, 768) dtype=float32>
  <tf.Variable 'tf_vi_t_model/vit/encoder/layer_._0/attention/attention/key/bias:0' shape=(768,) dtype=float32>
  

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling (Rescaling)       (None, 224, 224, 3)       0         
                                                                 
 permute (Permute)           (None, 3, 224, 224)       0         
                                                                 
 lambda (Lambda)             (None, 197, 768)          0         
                                                                 
 lambda_1 (Lambda)           (None, 768)               0         
                                                                 
 dense (Dense)               (None, 1)                 769       
                                                                 
Total params: 769 (3.00 KB)
Trainable params: 769 (3.00 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


# Download the data

In [ ]:
from huggingface_hub import snapshot_download

dataset_path = snapshot_download(
    repo_id="ggsmith/Hymenoptera",
    repo_type="dataset",
    local_dir=LOCAL_DIR
)

import tensorflow as tf
train_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_DIR + "/train",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_DIR + "/val",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

Fetching 400 files:   0%|          | 0/400 [00:00<?, ?it/s]

Found 245 files belonging to 2 classes.
Found 153 files belonging to 2 classes.


# Training

In [ ]:
#training again
h3 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/10
8/8 [==============================] - 37s 1s/step - loss: 0.3824 - accuracy: 0.8694 - val_loss: 0.2171 - val_accuracy: 0.9216
Epoch 2/10
8/8 [==============================] - 6s 734ms/step - loss: 0.1705 - accuracy: 0.9673 - val_loss: 0.1257 - val_accuracy: 0.9608
Epoch 3/10
8/8 [==============================] - 6s 784ms/step - loss: 0.1043 - accuracy: 0.9796 - val_loss: 0.0952 - val_accuracy: 0.9608
Epoch 4/10
8/8 [==============================] - 6s 744ms/step - loss: 0.0777 - accuracy: 0.9837 - val_loss: 0.0804 - val_accuracy: 0.9673
Epoch 5/10
8/8 [==============================] - 7s 804ms/step - loss: 0.0638 - accuracy: 0.9878 - val_loss: 0.0739 - val_accuracy: 0.9739
Epoch 6/10
8/8 [==============================] - 6s 765ms/step - loss: 0.0531 - accuracy: 0.9878 - val_loss: 0.0698 - val_accuracy: 0.9739
Epoch 7/10
8/8 [==============================] - 7s 807ms/step - loss: 0.0464 - accuracy: 0.9878 - val_loss: 0.0684 - val_accuracy: 0.9739
Epoch 8/10
8/8 [======